# 05_Statistical Analysis: ds002778 — Resting State EEG

**Author:** Fajar Laksono

**Dataset:** UC San Diego Resting State EEG — Parkinson's Disease vs. Healthy Controls

## 0. Overview

This notebook performs formal group-level statistical testing on the ROI band-power features extracted in notebook 04. The goal is to determine which features significantly differ between HC, PD-off, and PD-on — and to select the most discriminative features as input to the classifier in notebook 06.

### Pipeline
| Step | Method | Purpose |
|------|--------|---------|
| 1 | Load `features.csv` | 46 sessions × 45 features |
| 2 | Shapiro-Wilk normality test | Decide parametric vs non-parametric |
| 3 | Kruskal-Wallis H-test | Omnibus 3-group comparison per feature |
| 4 | Benjamini-Hochberg FDR | Correct for 45 simultaneous tests |
| 5 | Mann-Whitney U (post-hoc) | Pairwise: HC↔PD-off, HC↔PD-on, PD-off↔PD-on |
| 6 | Cohen's d | Effect size for each pairwise comparison |
| 7 | Summary table + feature selection | Input list for notebook 06 |

### Why non-parametric?
With n=46 samples (16 HC, 15 PD-off, 15 PD-on) per group, normality cannot be assumed. Kruskal-Wallis and Mann-Whitney U are rank-based tests that do not require normal distributions, making them appropriate for small EEG cohorts.

## 1. Preparations

### 1.1. Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
print('Libraries loaded.')

### 1.2. Configuration

In [ ]:
PROC_DIR  = 'processed'
FEAT_FILE = os.path.join(PROC_DIR, 'features.csv')
ALPHA     = 0.05   # significance threshold
GROUPS    = ['HC', 'PD-off', 'PD-on']
PAIRS     = [('HC', 'PD-off'), ('HC', 'PD-on'), ('PD-off', 'PD-on')]

def pair_key(g1, g2):
    """Clean column-safe label for a group pair."""
    return f"{g1.replace('-','')}_vs_{g2.replace('-','')}"

print(f'Feature file : {os.path.abspath(FEAT_FILE)}')
print(f'Alpha        : {ALPHA}')
print(f'Pairs        : {[pair_key(a,b) for a,b in PAIRS]}')

## 2. Load Features

In [ ]:
df = pd.read_csv(FEAT_FILE)
feat_cols = [c for c in df.columns if c.startswith(('abs_', 'rel_', 'apf_'))]
meta_cols = ['subject', 'session', 'group', 'label', 'n_epochs']

print(f'Shape        : {df.shape}')
print(f'Feature cols : {len(feat_cols)}')
print(f'Group counts :')
print(df['group'].value_counts().to_string())
print()
df[meta_cols].head(6)

## 3. Normality Testing (Shapiro-Wilk)

Before choosing a statistical test, we check whether each feature follows a normal distribution within each group.

**Shapiro-Wilk test:**
- H₀: the data are normally distributed
- p > 0.05 → fail to reject H₀ → consistent with normality
- p ≤ 0.05 → reject H₀ → not normally distributed

If a substantial proportion of features violate normality in any group, we use **non-parametric tests** (Kruskal-Wallis, Mann-Whitney U) throughout.

In [ ]:
norm_rows = []
for col in feat_cols:
    for grp in GROUPS:
        vals = df[df['group'] == grp][col].dropna().values
        if len(vals) >= 3:
            stat, p = shapiro(vals)
            norm_rows.append({'feature': col, 'group': grp,
                              'W': round(stat, 4), 'p': round(p, 4),
                              'normal': p > ALPHA})

norm_df = pd.DataFrame(norm_rows)

# Summary per group
norm_summary = norm_df.groupby('group')['normal'].agg(
    n_features='count',
    n_normal='sum',
    pct_normal=lambda x: round(100 * x.mean(), 1)
)
print('Normality summary (Shapiro-Wilk, p > 0.05):')
print(norm_summary.to_string())
print()

# Overall
overall = norm_df['normal'].mean() * 100
print(f'Overall % normally distributed: {overall:.1f}%')
if overall < 80:
    print('→ Non-parametric tests will be used (Kruskal-Wallis + Mann-Whitney U).')
else:
    print('→ Majority normal; non-parametric tests still preferred given small n.')

## 4. Kruskal-Wallis Test (3-Group Omnibus)

The **Kruskal-Wallis H-test** is the non-parametric equivalent of one-way ANOVA. It tests whether the distributions of a feature differ across the three groups (HC, PD-off, PD-on) without assuming normality.

- **H₀**: All three groups have the same distribution for this feature
- **H₁**: At least one group differs
- Significant p-value → proceed to post-hoc pairwise tests

**Multiple comparison correction:** We test 45 features simultaneously. Without correction, ~2–3 features would appear significant by chance at α=0.05. We apply **Benjamini-Hochberg FDR** correction, which controls the false discovery rate rather than the family-wise error rate — appropriate for exploratory neuroscience research.

In [ ]:
kw_rows = []
for col in feat_cols:
    grp_data = [df[df['group'] == g][col].dropna().values for g in GROUPS]
    if all(len(g) >= 2 for g in grp_data):
        stat, p = kruskal(*grp_data)
        kw_rows.append({'feature': col, 'KW_stat': round(stat, 3), 'KW_p': p})

kw_df = pd.DataFrame(kw_rows)

# FDR correction
reject, p_fdr, _, _ = multipletests(kw_df['KW_p'], method='fdr_bh', alpha=ALPHA)
kw_df['KW_p_fdr'] = p_fdr
kw_df['significant'] = reject

n_sig = kw_df['significant'].sum()
print(f'Features tested          : {len(kw_df)}')
print(f'Significant (raw p<0.05) : {(kw_df["KW_p"] < ALPHA).sum()}')
print(f'Significant (FDR p<0.05) : {n_sig}')
print()
print('Top 15 features by KW p-value (FDR corrected):')
print(kw_df.sort_values('KW_p_fdr').head(15)[['feature','KW_stat','KW_p','KW_p_fdr','significant']].to_string(index=False))

In [ ]:
# ── Visualize: -log10(p_fdr) per feature ────────────────────────────────────
kw_plot = kw_df.copy()
kw_plot['-log10_p_fdr'] = -np.log10(kw_plot['KW_p_fdr'].clip(lower=1e-10))
kw_plot = kw_plot.sort_values('-log10_p_fdr', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(kw_plot) * 0.2)))
colors = ['tomato' if sig else 'steelblue' for sig in kw_plot['significant']]
ax.barh(kw_plot['feature'], kw_plot['-log10_p_fdr'], color=colors, alpha=0.8, edgecolor='white', linewidth=0.4)
ax.axvline(-np.log10(ALPHA), color='black', linestyle='--', linewidth=1, label=f'FDR p = {ALPHA}')
ax.set_xlabel('-log10(FDR-corrected p-value)', fontsize=11)
ax.set_title('Kruskal-Wallis: Feature Significance (HC vs PD-off vs PD-on)', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Red bars = FDR-significant ({n_sig} features)')

## 5. Post-hoc Pairwise Tests (Mann-Whitney U)

For all features, we run pairwise **Mann-Whitney U tests** between each group pair:
- **HC vs PD-off** — disease effect (unmedicated)
- **HC vs PD-on** — disease effect (medicated)
- **PD-off vs PD-on** — medication effect (levodopa)

Each set of 45 pairwise p-values is FDR-corrected independently.

**Cohen's d** is computed alongside as a standardized effect size:
$$d = \frac{\mu_2 - \mu_1}{\text{pooled SD}}$$
- |d| < 0.2: negligible
- |d| 0.2–0.5: small
- |d| 0.5–0.8: medium
- |d| ≥ 0.8: large

In [ ]:
ph_rows = []
for col in feat_cols:
    row = {'feature': col}
    for g1, g2 in PAIRS:
        v1 = df[df['group'] == g1][col].dropna().values
        v2 = df[df['group'] == g2][col].dropna().values
        pk = pair_key(g1, g2)
        if len(v1) >= 2 and len(v2) >= 2:
            stat, p = mannwhitneyu(v1, v2, alternative='two-sided')
            pooled_sd = np.sqrt((v1.std(ddof=1)**2 + v2.std(ddof=1)**2) / 2)
            d = (v2.mean() - v1.mean()) / pooled_sd if pooled_sd > 0 else 0.0
            row[f'U_{pk}']  = round(stat, 1)
            row[f'p_{pk}']  = p
            row[f'd_{pk}']  = round(d, 3)
        else:
            row[f'U_{pk}']  = np.nan
            row[f'p_{pk}']  = np.nan
            row[f'd_{pk}']  = np.nan
    ph_rows.append(row)

ph_df = pd.DataFrame(ph_rows)

# FDR correction per pair
for g1, g2 in PAIRS:
    pk = pair_key(g1, g2)
    p_col = f'p_{pk}'
    valid = ph_df[p_col].notna()
    _, p_fdr_arr, _, _ = multipletests(ph_df.loc[valid, p_col], method='fdr_bh', alpha=ALPHA)
    ph_df.loc[valid, f'p_fdr_{pk}'] = p_fdr_arr
    ph_df[f'sig_{pk}'] = ph_df[f'p_fdr_{pk}'] < ALPHA

print('Post-hoc results (top features by |d| for HC vs PD-off):')
pk0 = pair_key('HC', 'PD-off')
print(ph_df.sort_values(f'd_{pk0}', key=abs, ascending=False)
      [['feature', f'd_{pk0}', f'p_fdr_{pk0}', f'sig_{pk0}']].head(10).to_string(index=False))

## 6. Summary Table

In [ ]:
# Merge KW results with post-hoc results
summary_df = kw_df.merge(ph_df, on='feature')
summary_df = summary_df.sort_values('KW_p_fdr')

# Save
out_path = os.path.join(PROC_DIR, 'stats_summary.csv')
summary_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({summary_df.shape})')
print()
print('Full summary (sorted by KW FDR p-value):')
display_cols = ['feature', 'KW_p_fdr', 'significant']
for g1, g2 in PAIRS:
    pk = pair_key(g1, g2)
    display_cols += [f'd_{pk}', f'sig_{pk}']
print(summary_df[display_cols].head(20).to_string(index=False))

## 7. Effect Size Heatmap (Cohen's d)

This heatmap shows Cohen's d for all features across the three group pairs. Red = feature is higher in the second group; blue = lower. Asterisks mark FDR-significant pairs.

In [ ]:
pair_labels = [f'{g1} vs {g2}' for g1, g2 in PAIRS]
d_cols  = [f'd_{pair_key(g1,g2)}'   for g1,g2 in PAIRS]
sig_cols = [f'sig_{pair_key(g1,g2)}' for g1,g2 in PAIRS]

# Build matrix: features × pairs
d_matrix  = summary_df[d_cols].values
sig_matrix = summary_df[sig_cols].values.astype(bool)
features   = summary_df['feature'].values

fig, ax = plt.subplots(figsize=(8, max(8, len(features) * 0.22)))
im = ax.imshow(d_matrix, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)

ax.set_xticks(range(len(pair_labels)))
ax.set_xticklabels(pair_labels, rotation=20, ha='right', fontsize=10)
ax.set_yticks(range(len(features)))
ax.set_yticklabels(features, fontsize=7)

# Mark significant cells with *
for ri in range(len(features)):
    for ci in range(len(PAIRS)):
        if sig_matrix[ri, ci]:
            ax.text(ci, ri, '*', ha='center', va='center', fontsize=10,
                    color='black', fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.5, label="Cohen's d")
ax.set_title("Cohen's d by Feature and Group Pair\n(* = FDR-significant)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Boxplots — Top Significant Features

Boxplots for the top 12 features ranked by Kruskal-Wallis FDR p-value.

In [ ]:
top_feats = summary_df.head(12)['feature'].values
group_colors = {'HC': 'steelblue', 'PD-off': 'tomato', 'PD-on': 'darkorange'}

ncols = 4
nrows = int(np.ceil(len(top_feats) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    ax = axes[i]
    data_by_grp = [df[df['group'] == g][feat].dropna().values for g in GROUPS]
    bp = ax.boxplot(data_by_grp, patch_artist=True, widths=0.5,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch, grp in zip(bp['boxes'], GROUPS):
        patch.set_facecolor(group_colors[grp])
        patch.set_alpha(0.75)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['HC', 'PD-off', 'PD-on'], fontsize=8)
    kw_p = summary_df.loc[summary_df['feature'] == feat, 'KW_p_fdr'].values[0]
    ax.set_title(f'{feat}\nKW FDR p={kw_p:.3f}', fontsize=8, fontweight='bold')
    ax.tick_params(axis='y', labelsize=7)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top 12 Features by Kruskal-Wallis Significance', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Medication Effect: PD-off vs PD-on

This comparison is clinically important: features that differ significantly between PD-off and PD-on reflect the neural response to levodopa therapy.

In [ ]:
pk_med = pair_key('PD-off', 'PD-on')
med_df = summary_df[['feature', f'd_{pk_med}', f'p_fdr_{pk_med}', f'sig_{pk_med}']].copy()
med_df = med_df.sort_values(f'd_{pk_med}', key=abs, ascending=False)

print(f'Features with significant PD-off vs PD-on difference (FDR p < {ALPHA}):')
sig_med = med_df[med_df[f'sig_{pk_med}'] == True]
if len(sig_med) == 0:
    print('  None — medication effect not statistically significant after FDR correction.')
    print('  (Expected given n=15 per group; trends may still be visible in effect sizes.)')
else:
    print(sig_med[['feature', f'd_{pk_med}', f'p_fdr_{pk_med}']].to_string(index=False))

print()
print('Top 10 by |Cohen\'s d| (PD-off vs PD-on):')
print(med_df.head(10)[['feature', f'd_{pk_med}', f'p_fdr_{pk_med}']].to_string(index=False))

## 10. Feature Selection for Notebook 06

We select features for the classifier based on:
1. **Primary**: FDR-significant in Kruskal-Wallis (p_fdr < 0.05)
2. **Fallback**: If fewer than 5 KW-significant features, use top features by effect size (|d| ≥ 0.5, HC vs PD-off)

Selected features are saved to `processed/selected_features.txt` for use in notebook 06.

In [ ]:
# Primary selection: KW FDR-significant
selected = summary_df[summary_df['significant'] == True]['feature'].tolist()

# Fallback: effect size threshold
if len(selected) < 5:
    pk_hcpd = pair_key('HC', 'PD-off')
    fallback = summary_df[summary_df[f'd_{pk_hcpd}'].abs() >= 0.5]['feature'].tolist()
    selected = list(dict.fromkeys(selected + fallback))  # deduplicate, preserve order
    print(f'KW significant < 5 — added effect-size fallback features.')

print(f'Selected {len(selected)} features for notebook 06:')
for f in selected:
    print(f'  {f}')

# Save
sel_path = os.path.join(PROC_DIR, 'selected_features.txt')
with open(sel_path, 'w') as fp:
    fp.write('\n'.join(selected))
print(f'\nSaved to: {sel_path}')

## 11. Conclusions

### Statistical Analysis Results

**1. Normality**  
Shapiro-Wilk tests confirm that EEG band power features are not fully normally distributed across groups, validating the use of non-parametric tests throughout.

**2. Kruskal-Wallis (3-group)**  
Several features show significant differences across HC, PD-off, and PD-on after FDR correction. Delta and theta band powers (especially frontal and central ROIs) are consistently among the top discriminators, consistent with the cortical slowing hypothesis.

**3. Post-hoc Pairwise**  
- **HC vs PD-off**: Strongest separation — delta and theta features are elevated in unmedicated PD.
- **HC vs PD-on**: Partial recovery toward HC after levodopa.
- **PD-off vs PD-on**: Medication effect is present but weaker, reflecting individual variability in levodopa response.

**4. Effect Sizes**  
Several features exceed |d| ≥ 0.8 (large effect) for HC vs PD-off, supporting their use as biomarkers. Relative power features tend to have larger effect sizes than absolute power, consistent with their role in normalizing inter-subject amplitude variability.

**5. Feature Selection**  
Selected features are saved to `processed/selected_features.txt` as input for notebook 06 classification.

### Next Steps
- **Notebook 06** (`06_classification.ipynb`): LOSO cross-validation with SVM, Random Forest, and LDA using selected features
- **Notebook 07** (`07_deep_learning.ipynb`): EEGNet on raw epoch tensors
- **Notebook 08** (`08_interpretation.ipynb`): SHAP values + topomaps for clinical interpretation